In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import RobustScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D3 
clinical_train.isnull().sum().sum()

0

### COX assumption in Train data

In [6]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic    p  -log2(p)
LBP_003_CT                                                       0.01 0.91      0.13
LBP_003_PET                                                      0.00 0.98      0.03
LBP_012_CT                                                       0.00 0.95      0.08
LBP_012_PET                                                      0.00 0.97      0.04
LBP_021_CT                                                       0.00 0.97      0.05
LBP_021_PET                                                      0.01 0.92      0.12
LBP_030_CT                                                       0.00 0.99      0.02
LBP_030_PET       

In [7]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


In [8]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic      p  -log2(p)
LBP_003_CT                                                       0.06   0.81      0.31
LBP_003_PET                                                      0.76   0.38      1.38
LBP_012_CT                                                       0.13   0.72      0.48
LBP_012_PET                                                      0.27   0.60      0.73
LBP_021_CT                                                       0.56   0.45      1.14
LBP_021_PET                                                      0.00   0.97      0.04
LBP_030_CT                                                       0.04   0.83      0.26
LB

In [9]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['charlson', 'glszm_LargeAreaEmphasis_CT_c16',
       'glszm_LargeAreaLowGrayLevelEmphasis_CT_c16',
       'glszm_ZoneVariance_CT_c16', 'ngtdm_Busyness_d_1_PET_b2'],
      dtype='object')


## Test dataset: MAASTRO 

In [10]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [11]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [12]:
# Need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [13]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [14]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [15]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [16]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# Set y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [17]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [18]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [19]:
# Change the name of a column 'DFS_event' in the clincial_test 
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [20]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

# Standardization

In [21]:
# Copy the original X for later 
original_X = X.copy()

In [22]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']
### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
scaler = RobustScaler()  
X_numeric_std = scaler.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [23]:
# Divide the X_MAASTRO into numerical part and categorical part 
X_MAASTRO_categorical = X_MAASTRO[categorical_columns]
X_MAASTRO_numeric = X_MAASTRO.drop(categorical_columns, axis=1)

# Save the column name and index of the numeric part
X_MAASTRO_numeric_columns = X_MAASTRO_numeric.columns
X_MAASTRO_numeric_index = X_MAASTRO_numeric.index

In [24]:
# Standardize the numeric part 
X_MAASTRO_numeric_std = scaler.transform(X_MAASTRO_numeric)

# Change the standardized part into a dataframe 
X_MAASTRO_numeric_std = pd.DataFrame(X_MAASTRO_numeric_std, columns=X_MAASTRO_numeric_columns, index=X_MAASTRO_numeric_index)

# Concat the standardized part with the categorical part 
X_MAASTRO_std = pd.concat([X_MAASTRO_categorical, X_MAASTRO_numeric_std], axis=1)

# Change the column order of X_MAASTRO_std
X_MAASTRO_std = X_MAASTRO_std[original_X.columns]

In [25]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = X_MAASTRO_std 

In [26]:
X_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_003_PET,LBP_012_PET,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,...,0.000123,0.001172,0.029974,0.815962,0.000000,0.002035,0.140311,0.000062,0.009991,0.000370
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,...,0.000349,0.008383,0.059728,0.707300,0.000000,0.008732,0.191058,0.000349,0.023402,0.000699
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,...,0.000000,0.001759,0.039463,0.819828,0.000034,0.002518,0.126531,0.000000,0.009452,0.000414
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,...,0.000300,0.006892,0.061133,0.716212,0.000000,0.003296,0.192388,0.000000,0.018879,0.000899
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,...,0.000000,0.001993,0.058589,0.696493,0.000199,0.008968,0.202073,0.000399,0.029892,0.001395
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,...,0.000000,0.001562,0.029444,0.804831,0.000000,0.001442,0.152626,0.000000,0.009734,0.000361
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,...,0.000039,0.001968,0.046451,0.792151,0.000000,0.004094,0.142778,0.000079,0.011810,0.000630
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,...,0.000000,0.000326,0.016316,0.832202,0.000000,0.000914,0.140582,0.000000,0.009137,0.000522
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,...,0.000000,0.001030,0.038862,0.787588,0.000000,0.003144,0.156640,0.000054,0.011978,0.000705


In [27]:
X_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_003_PET,LBP_012_PET,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET
0,-0.497635,1,0,1,0,0,1,0.0,0,-0.747766,...,1.259511,-0.187895,-0.237601,0.332001,0.000000,-0.255811,-0.228027,0.038290,-0.362826,-0.200426
1,-0.473435,0,0,0,0,1,0,0.0,1,0.159774,...,3.566490,3.137073,0.913224,-1.314176,0.000000,2.591499,1.070407,1.827361,1.866886,0.869468
2,-0.113739,0,1,0,0,0,1,0.0,1,0.610629,...,0.000000,0.082981,0.129427,0.390580,0.524263,-0.050487,-0.580607,-0.345359,-0.452526,-0.057457
3,0.826312,0,0,0,0,1,0,0.0,1,0.494088,...,3.059892,2.449854,0.967575,-1.179161,0.000000,0.280373,1.104438,-0.345359,1.114908,1.522258
4,0.596634,0,0,0,0,1,0,0.0,1,1.007387,...,0.000000,0.190665,0.869189,-1.477904,3.028668,2.691677,1.352218,2.133915,2.946000,3.137491
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,0.000000,0,0,1,0,0,1,1.0,0,-0.747766,...,0.000000,-0.007844,-0.258121,0.163381,0.000000,-0.508000,0.087068,-0.345359,-0.405549,-0.231420
135,0.671213,0,0,1,0,0,1,1.0,0,-0.747766,...,0.401955,0.179343,0.399712,-0.028726,0.000000,0.619505,-0.164891,0.144386,-0.060514,0.645662
136,-0.235838,0,0,1,0,0,1,1.0,1,0.560275,...,0.000000,-0.577753,-0.765863,0.578039,0.000000,-0.732668,-0.221085,-0.345359,-0.504845,0.294840
137,0.421516,0,0,1,0,0,1,1.0,1,1.620942,...,0.000000,-0.253379,0.106163,-0.097845,0.000000,0.215431,0.189762,-0.008205,-0.032466,0.889136


In [28]:
MAASTRO_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_003_PET,LBP_012_PET,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET
0,55,0,0,1,0,0,1,1,1,0,...,0.000026,0.001181,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341
1,55,0,0,1,0,0,0,0,0,20,...,0.000056,0.002735,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335
2,55,0,0,1,0,0,0,0,1,6,...,0.000286,0.001888,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229
3,61,1,0,0,0,1,1,0,1,45,...,0.000160,0.003037,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240
4,70,0,0,1,0,0,1,1,1,59,...,0.000071,0.000881,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,66,1,0,0,0,1,0,0,0,55,...,0.000000,0.000544,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078
95,63,0,0,0,0,1,0,0,1,174,...,0.000109,0.000869,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489
96,63,0,0,1,0,0,1,1,1,0,...,0.000000,0.000734,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141
97,54,0,0,1,0,0,1,1,0,0,...,0.000114,0.001640,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038


In [29]:
MAASTRO_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_003_PET,LBP_012_PET,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET
0,-0.436476,0,0,1,0,0,1,1,1,-0.747766,...,0.267896,-0.183833,-0.282721,0.656586,0.398737,-0.396081,-0.691181,-0.182156,-0.645585,-0.294798
1,-0.436476,0,0,1,0,0,0,0,0,-0.085444,...,0.569866,0.532732,0.522080,0.193848,2.544568,0.136471,-0.518031,0.696135,-0.669273,-0.315036
2,-0.436476,0,0,1,0,0,0,0,1,-0.549070,...,2.921562,0.142520,-0.642200,0.548796,0.869691,-0.342585,-0.305528,-0.345359,-0.587357,-0.660121
3,0.045320,1,0,0,0,1,1,0,1,0.742458,...,1.632171,0.672170,0.426924,-0.501409,0.000000,0.407999,0.572422,0.151803,0.168550,-0.624716
4,0.768012,0,0,1,0,0,1,1,1,1.206084,...,0.720063,-0.321768,-0.082258,0.466887,0.000000,-0.221668,-0.539593,-0.126027,-0.464627,0.316889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,0.446816,1,0,0,0,1,0,0,0,1.073619,...,0.000000,-0.477333,-0.720496,0.614215,0.000000,-0.559327,-0.307777,0.138162,-0.576570,-1.152435
95,0.205918,0,0,0,0,1,0,0,1,5.014436,...,1.109092,-0.327551,-0.237418,0.414854,0.000000,-0.890248,-0.296871,-0.007528,-0.543158,0.186261
96,0.205918,0,0,1,0,0,1,1,1,-0.747766,...,0.000000,-0.389935,-0.725739,0.998508,0.000000,-0.701251,-0.873097,-0.169833,-1.076336,-0.946098
97,-0.516775,0,0,1,0,0,1,1,0,-0.747766,...,1.168246,0.027935,-0.414541,0.815980,0.000000,-0.488772,-0.807734,-0.345359,-1.066559,-1.281379


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [30]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 18:26:33,017] A new study created in memory with name: no-name-fc98e6e7-e390-4ad2-870c-488a75946adf


  0%|          | 0/1 [00:00<?, ?it/s]

[W 2024-04-16 18:26:33,556] Trial 0 failed with parameters: {} because of the following error: ValueError('search direction contains NaN or infinite values').
Traceback (most recent call last):
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/64/jkqp6xyx2hj50dmd2pqfm3780000gn/T/ipykernel_24617/1570569369.py", line 62, in objective
    model.fit(X_train_std, y_train)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/sksurv/linear_model/coxph.py", line 454, in fit
    raise ValueError("search direction contains NaN or infinite values")
ValueError: search direction contains NaN or infinite values
[W 2024-04-16 18:26:33,559] Trial 0 failed with value None.


ValueError: search direction contains NaN or infinite values

In [31]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

In [32]:
# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

ValueError: No trials are completed yet.

In [33]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

ValueError: No trials are completed yet.

#### Test

In [34]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [35]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

ValueError: search direction contains NaN or infinite values

In [36]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [37]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

NameError: name 'c_index' is not defined

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [38]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 18:27:48,013] A new study created in memory with name: no-name-95b1697c-fde7-43d7-adba-3d2bc081014b


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.41832669322709165
Fold 2 C-index: 0.6647286821705426
Fold 3 C-index: 0.6468085106382979
Fold 4 C-index: 0.55893536121673


[I 2024-04-16 18:27:48,356] A new study created in memory with name: no-name-9254973c-8a2d-465f-a24c-1d7a6c046ac0


Fold 5 C-index: 0.5064377682403434
[I 2024-04-16 18:27:48,352] Trial 0 finished with value: 0.559047403098601 and parameters: {}. Best is trial 0 with value: 0.559047403098601.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.559047403098601], datetime_start=datetime.datetime(2024, 4, 16, 18, 27, 48, 30722), datetime_complete=datetime.datetime(2024, 4, 16, 18, 27, 48, 352053), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.559047403098601


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2472471085139047
Fold 2 IBS: 0.23203988104602738
Fold 3 IBS: 0.22898186657343633
Fold 4 IBS: 0.24197478013193247
Fold 5 IBS: 0.22939559171499582
[I 2024-04-16 18:27:48,707] Trial 0 finished with value: 0.23592784559605934 and parameters: {}. Best is trial 0 with value: 0.23592784559605934.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784559605934], datetime_start=datetime.datetime(2024, 4, 16, 18, 27, 48, 372242), datetime_complete=datetime.datetime(2024, 4, 16, 18, 27, 48, 707546), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784559605934


In [39]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [40]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.559
train_ibs:  0.236


#### Test

In [41]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [42]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.546


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [43]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [44]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 18:27:48,815] A new study created in memory with name: no-name-7afe07d1-b634-4759-a419-a2effa3d5abd


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.532319391634981


[I 2024-04-16 18:27:49,173] A new study created in memory with name: no-name-28c136c5-9d44-4a58-a1b8-dc81b9660ea0


Fold 5 C-index: 0.48068669527896996
[I 2024-04-16 18:27:49,170] Trial 0 finished with value: 0.5738322950513342 and parameters: {}. Best is trial 0 with value: 0.5738322950513342.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5738322950513342], datetime_start=datetime.datetime(2024, 4, 16, 18, 27, 48, 834464), datetime_complete=datetime.datetime(2024, 4, 16, 18, 27, 49, 170593), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5738322950513342


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2474289394458287
Fold 2 IBS: 0.231967597908397
Fold 3 IBS: 0.22852058423016272
Fold 4 IBS: 0.24214585156595497
Fold 5 IBS: 0.22935161129821743
[I 2024-04-16 18:27:49,575] Trial 0 finished with value: 0.23588291688971216 and parameters: {}. Best is trial 0 with value: 0.23588291688971216.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23588291688971216], datetime_start=datetime.datetime(2024, 4, 16, 18, 27, 49, 193288), datetime_complete=datetime.datetime(2024, 4, 16, 18, 27, 49, 575327), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23588291688971216


In [45]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [46]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.574
train_ibs:  0.236


#### Test

In [47]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [48]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.562


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.229


In [49]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [50]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 18:27:49,687] A new study created in memory with name: no-name-02c317c2-39e3-4211-bada-aebfd97f22c7


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.532319391634981
Fold 5 C-index: 0.48068669527896996
[I 2024-04-16 18:27:50,060] Trial 0 finished with value: 0.5738322950513342 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.5738322950513342.
Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.532319391634981
Fold 5 C-index: 0.48068669527896996
[I 2024-04-16 18:27:50,440] Trial 1 finished with value: 0.5738322950513342 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.5738322950513342.
Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.532319391634981
Fold 5 C-index: 0.48068669527896996
[I 2024-04-16 18:27:50,773] Trial 2 finished with value: 0.5738322950513342 and parameters: {'l1_ratio': 0.2269287684188466

Fold 5 C-index: 0.48068669527896996
[I 2024-04-16 18:27:57,285] Trial 23 finished with value: 0.5738322950513342 and parameters: {'l1_ratio': 0.2976235432977797}. Best is trial 0 with value: 0.5738322950513342.
Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.532319391634981
Fold 5 C-index: 0.48068669527896996
[I 2024-04-16 18:27:57,596] Trial 24 finished with value: 0.5738322950513342 and parameters: {'l1_ratio': 0.15433706302991543}. Best is trial 0 with value: 0.5738322950513342.
Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.532319391634981
Fold 5 C-index: 0.48068669527896996
[I 2024-04-16 18:27:57,901] Trial 25 finished with value: 0.5738322950513342 and parameters: {'l1_ratio': 0.4626599986704694}. Best is trial 0 with value: 0.5738322950513342.
Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index:

Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.532319391634981
Fold 5 C-index: 0.48068669527896996
[I 2024-04-16 18:28:04,878] Trial 47 finished with value: 0.5738322950513342 and parameters: {'l1_ratio': 0.8434660181559295}. Best is trial 0 with value: 0.5738322950513342.
Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.532319391634981
Fold 5 C-index: 0.48068669527896996
[I 2024-04-16 18:28:05,180] Trial 48 finished with value: 0.5738322950513342 and parameters: {'l1_ratio': 0.5675456270820948}. Best is trial 0 with value: 0.5738322950513342.
Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.532319391634981
Fold 5 C-index: 0.48068669527896996
[I 2024-04-16 18:28:05,473] Trial 49 finished with value: 0.5738322950513342 and parameters: {'l1_ratio': 0.71531903387609

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.2474287960786108
Fold 2 IBS: 0.23196765088486512
Fold 3 IBS: 0.22852715485205996
Fold 4 IBS: 0.2421457297130617
Fold 5 IBS: 0.22935166169383375
[I 2024-04-16 18:28:24,557] Trial 0 finished with value: 0.23588419864448626 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.23588419864448626.
Fold 1 IBS: 0.24742812136880135
Fold 2 IBS: 0.23196790029317935
Fold 3 IBS: 0.22855568720417294
Fold 4 IBS: 0.2421451560877283
Fold 5 IBS: 0.22935189846717802
[I 2024-04-16 18:28:24,905] Trial 1 finished with value: 0.235889752684212 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.23588419864448626.
Fold 1 IBS: 0.2474278234473111
Fold 2 IBS: 0.2319680104690308
Fold 3 IBS: 0.22856716662614762
Fold 4 IBS: 0.24214490271345115
Fold 5 IBS: 0.22935200280729984
[I 2024-04-16 18:28:25,474] Trial 2 finished with value: 0.2358919812126481 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.23588419864448626

Fold 1 IBS: 0.2474289071342513
Fold 2 IBS: 0.23196760984743806
Fold 3 IBS: 0.22852208195931123
Fold 4 IBS: 0.2421458241042707
Fold 5 IBS: 0.22935162265877537
[I 2024-04-16 18:28:34,602] Trial 25 finished with value: 0.23588320914080932 and parameters: {'l1_ratio': 0.9106143729430608}. Best is trial 11 with value: 0.2358829251943022.
Fold 1 IBS: 0.247428824861718
Fold 2 IBS: 0.23196764024849678
Fold 3 IBS: 0.22852585106567655
Fold 4 IBS: 0.24214575417783435
Fold 5 IBS: 0.2293516515785325
[I 2024-04-16 18:28:35,021] Trial 26 finished with value: 0.23588394438645163 and parameters: {'l1_ratio': 0.7417169602645222}. Best is trial 11 with value: 0.2358829251943022.
Fold 1 IBS: 0.2474287619017622
Fold 2 IBS: 0.2319676635147703
Fold 3 IBS: 0.22852869312563417
Fold 4 IBS: 0.2421457006631275
Fold 5 IBS: 0.22935167370312118
[I 2024-04-16 18:28:35,484] Trial 27 finished with value: 0.2358844985816831 and parameters: {'l1_ratio': 0.6494728799686162}. Best is trial 11 with value: 0.2358829251943022

Fold 3 IBS: 0.22852268865983366
Fold 4 IBS: 0.24214581292783316
Fold 5 IBS: 0.2293516272818253
[I 2024-04-16 18:28:45,273] Trial 50 finished with value: 0.23588332751202779 and parameters: {'l1_ratio': 0.8786447363305766}. Best is trial 40 with value: 0.23588291752493645.
Fold 1 IBS: 0.24742892194226343
Fold 2 IBS: 0.23196760437587396
Fold 3 IBS: 0.22852139680290473
Fold 4 IBS: 0.2421458366897119
Fold 5 IBS: 0.2293516174525529
[I 2024-04-16 18:28:45,784] Trial 51 finished with value: 0.23588307545266138 and parameters: {'l1_ratio': 0.9495136153828908}. Best is trial 40 with value: 0.23588291752493645.
Fold 1 IBS: 0.24742892416440054
Fold 2 IBS: 0.2319676035548001
Fold 3 IBS: 0.22852129380614417
Fold 4 IBS: 0.2421458385783117
Fold 5 IBS: 0.2293516166712635
[I 2024-04-16 18:28:46,227] Trial 52 finished with value: 0.235883055354984 and parameters: {'l1_ratio': 0.9556391205035538}. Best is trial 40 with value: 0.23588291752493645.
Fold 1 IBS: 0.24742893595289045
Fold 2 IBS: 0.231967599199

Fold 2 IBS: 0.23196760802321334
Fold 3 IBS: 0.2285218537600512
Fold 4 IBS: 0.24214582830026468
Fold 5 IBS: 0.22935162092305858
[I 2024-04-16 18:28:55,730] Trial 75 finished with value: 0.2358831646155645 and parameters: {'l1_ratio': 0.9232248440809541}. Best is trial 40 with value: 0.23588291752493645.
Fold 1 IBS: 0.24742892834880997
Fold 2 IBS: 0.2319676020086762
Fold 3 IBS: 0.22852109972961704
Fold 4 IBS: 0.2421458421346435
Fold 5 IBS: 0.22935161520003225
[I 2024-04-16 18:28:56,154] Trial 76 finished with value: 0.2358830174843558 and parameters: {'l1_ratio': 0.9673906281038791}. Best is trial 40 with value: 0.23588291752493645.
Fold 1 IBS: 0.24742890221953948
Fold 2 IBS: 0.2319676116634415
Fold 3 IBS: 0.22852230889959982
Fold 4 IBS: 0.242145819927191
Fold 5 IBS: 0.22935162438662726
[I 2024-04-16 18:28:56,591] Trial 77 finished with value: 0.2358832534192798 and parameters: {'l1_ratio': 0.8983978427418612}. Best is trial 40 with value: 0.23588291752493645.
Fold 1 IBS: 0.2474289159174

In [51]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [52]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.574
train_ibs:  0.236


#### Test

In [53]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [54]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6964995386793018)

test_cindex : 0.562


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.9997874437591321)

test_ibs:  0.229


In [55]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [56]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 18:29:05,580] A new study created in memory with name: no-name-bfa4fafd-4411-44c8-9b04-8598dc4cf2a7


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.751937984496124
Fold 3 C-index: 0.6680851063829787
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.6566523605150214
[I 2024-04-16 18:29:18,816] Trial 0 finished with value: 0.6887600217317204 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6887600217317204.
Fold 1 C-index: 0.5418326693227091
Fold 2 C-index: 0.6550387596899225
Fold 3 C-index: 0.7191489361702128
Fold 4 C-index: 0.6349809885931559
Fold 5 C-index: 0.5321888412017167
[I 2024-04-16 18:29:21,198] Trial 1 finished with value: 0.6166380389955435 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 1 C-index: 0.49800796812749004
Fold 2 C-index: 0.810077519379845
Fold 3 C-index: 0.851063829787234
Fold 4 C-index: 0.7642585551330798
Fold 5 C-index: 0.7467811158798283
[I 2024-04-16 18:29:56,302] Trial 17 finished with value: 0.7340377976614955 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 8, 'min_samples_leaf': 16, 'max_depth': 7, 'n_estimators': 96, 'oob_score': True, 'max_samples': 0.9816256083986692, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.10458249310616385, 'warm_start': True}. Best is trial 17 with value: 0.7340377976614955.
Fold 1 C-index: 0.4820717131474104
Fold 2 C-index: 0.7945736434108527
Fold 3 C-index: 0.8297872340425532
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.6652360515021459
[I 2024-04-16 18:29:56,631] Trial 18 finished with value: 0.6851322075080448 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 9, 'min_samples_leaf': 15, 'max_depth': 1, 'n_estimators': 105, 'oob_score': True, 'max_samples': 0.9862374149206906,

Fold 1 C-index: 0.5219123505976095
Fold 2 C-index: 0.8062015503875969
Fold 3 C-index: 0.8808510638297873
Fold 4 C-index: 0.8136882129277566
Fold 5 C-index: 0.8454935622317596
[I 2024-04-16 18:30:07,476] Trial 32 finished with value: 0.7736293479949021 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 12, 'n_estimators': 241, 'oob_score': True, 'max_samples': 0.717337635186249, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.11772473625239274, 'warm_start': True}. Best is trial 27 with value: 0.7847011373516671.
Fold 1 C-index: 0.5099601593625498
Fold 2 C-index: 0.8527131782945736
Fold 3 C-index: 0.9063829787234042
Fold 4 C-index: 0.8745247148288974
Fold 5 C-index: 0.9141630901287554
[I 2024-04-16 18:30:09,051] Trial 33 finished with value: 0.811548824267636 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 17, 'min_samples_leaf': 6, 'max_depth': 12, 'n_estimators': 272, 'oob_score': True, 'max_samples': 0.7247276784505238,

...

Fold 1 IBS: 0.22149490022552124
Fold 2 IBS: 0.18466865634717505
Fold 3 IBS: 0.23548539432841317
Fold 4 IBS: 0.2113697424267566
Fold 5 IBS: 0.2119677581906565
[I 2024-04-16 18:53:22,047] Trial 32 finished with value: 0.2129972903037045 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 294, 'oob_score': True, 'max_samples': 0.9673759674409311, 'max_features': None, 'min_weight_fraction_leaf': 0.27417727195073677}. Best is trial 32 with value: 0.2129972903037045.
Fold 1 IBS: 0.2567229873354098
Fold 2 IBS: 0.18854079292396503
Fold 3 IBS: 0.2249457788925835
Fold 4 IBS: 0.22508129343804778
Fold 5 IBS: 0.21386415001842546
[I 2024-04-16 18:53:53,472] Trial 33 finished with value: 0.2218310005216863 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 13, 'max_depth': 13, 'n_estimators': 304, 'oob_score': True, 'max_samples': 0.9964442664551478, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.23233061602902896
Fold 2 IBS: 0.20470776398952695
Fold 3 IBS: 0.24473722549758978
Fold 4 IBS: 0.23554082329183057
Fold 5 IBS: 0.22266798632628182
[I 2024-04-16 18:55:50,400] Trial 48 finished with value: 0.22799688302685164 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 9, 'min_samples_leaf': 9, 'max_depth': 14, 'n_estimators': 277, 'oob_score': True, 'max_samples': 0.7486071388218823, 'max_features': None, 'min_weight_fraction_leaf': 0.3316489763494004}. Best is trial 43 with value: 0.2123917039278164.
Fold 1 IBS: 0.22140751210407167
Fold 2 IBS: 0.18446093935852728
Fold 3 IBS: 0.23819663142832964
Fold 4 IBS: 0.21304733351095642
Fold 5 IBS: 0.21354093321412873
[I 2024-04-16 18:56:07,866] Trial 49 finished with value: 0.21413066992320276 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 18, 'min_samples_leaf': 20, 'max_depth': 11, 'n_estimators': 373, 'oob_score': True, 'max_samples': 0.9104681524496127, 'max_features': None, 'min_weight_fraction_leaf

Fold 1 IBS: 0.2365430487443625
Fold 2 IBS: 0.18238280317098843
Fold 3 IBS: 0.2322276544017584
Fold 4 IBS: 0.21380685041046066
Fold 5 IBS: 0.21115528132467798
[I 2024-04-16 19:01:04,218] Trial 64 finished with value: 0.21522312761044962 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 18, 'min_samples_leaf': 17, 'max_depth': 9, 'n_estimators': 405, 'oob_score': True, 'max_samples': 0.9369908825823273, 'max_features': None, 'min_weight_fraction_leaf': 0.19119487575645203}. Best is trial 43 with value: 0.2123917039278164.
Fold 1 IBS: 0.2545794604503248
Fold 2 IBS: 0.18515187912067008
Fold 3 IBS: 0.22861527937785475
Fold 4 IBS: 0.22050482646274178
Fold 5 IBS: 0.2129134229864224
[I 2024-04-16 19:01:39,459] Trial 65 finished with value: 0.22035297367960277 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 15, 'max_depth': 6, 'n_estimators': 362, 'oob_score': True, 'max_samples': 0.9998359499913109, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.2511057207255618
Fold 2 IBS: 0.21572593535559278
Fold 3 IBS: 0.2149073004231141
Fold 4 IBS: 0.2472468833843753
Fold 5 IBS: 0.24247425822381166
[I 2024-04-16 19:05:35,428] Trial 80 finished with value: 0.23429201962249113 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 16, 'max_depth': 12, 'n_estimators': 335, 'oob_score': True, 'max_samples': 0.3618038809495957, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.15390327148267055}. Best is trial 43 with value: 0.2123917039278164.
Fold 1 IBS: 0.22731065610063553
Fold 2 IBS: 0.18233470119562936
Fold 3 IBS: 0.23388585350331287
Fold 4 IBS: 0.21305351886000728
Fold 5 IBS: 0.21153134197254575
[I 2024-04-16 19:05:58,592] Trial 81 finished with value: 0.21362321432642614 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 17, 'min_samples_leaf': 18, 'max_depth': 11, 'n_estimators': 318, 'oob_score': True, 'max_samples': 0.9436804929256887, 'max_features': None, 'min_weight_fraction_l

Fold 1 IBS: 0.21975345965751325
Fold 2 IBS: 0.1826687616539912
Fold 3 IBS: 0.23593232323623828
Fold 4 IBS: 0.21692323202986982
Fold 5 IBS: 0.2157946933853038
[I 2024-04-16 19:09:36,847] Trial 96 finished with value: 0.21421449399258327 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 9, 'min_samples_leaf': 20, 'max_depth': 12, 'n_estimators': 296, 'oob_score': True, 'max_samples': 0.8401991280225705, 'max_features': None, 'min_weight_fraction_leaf': 0.20291905202477475}. Best is trial 85 with value: 0.21229918040313978.
Fold 1 IBS: 0.22595559817931396
Fold 2 IBS: 0.18811290557691393
Fold 3 IBS: 0.23790732230802805
Fold 4 IBS: 0.22652570350197412
Fold 5 IBS: 0.21929538367243218
[I 2024-04-16 19:09:50,533] Trial 97 finished with value: 0.21955938264773245 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 7, 'min_samples_leaf': 19, 'max_depth': 15, 'n_estimators': 270, 'oob_score': True, 'max_samples': 0.9001142343235446, 'max_features': None, 'min_weight_fraction_leaf'

In [57]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [58]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.847
train_ibs:  0.212


#### Test

In [59]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

In [60]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=16, max_features='auto', max_leaf_nodes=14,
                     max_samples=0.9873533850424658, min_samples_leaf=4,
                     min_samples_split=8,
                     min_weight_fraction_leaf=0.04287202412707603,
                     n_estimators=441, random_state=123, warm_start=True)

test_cindex:  0.54


RandomSurvivalForest(max_depth=13, max_features=None, max_leaf_nodes=11,
                     max_samples=0.9290276329324785, min_samples_leaf=20,
                     min_samples_split=9,
                     min_weight_fraction_leaf=0.20767416366171196,
                     n_estimators=268, oob_score=True, random_state=123)

test_ibs:  0.262


In [61]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [62]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [63]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 19:10:21,288] A new study created in memory with name: no-name-35bf5694-0822-4787-a4f2-09848870b61c


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.50199203187251
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.8425531914893617
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.7381974248927039
[I 2024-04-16 19:10:22,253] Trial 0 finished with value: 0.716790225055755 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.716790225055755.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 19:10:24,191] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. B

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 19:10:46,494] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 14, 'min_samples_leaf': 20, 'max_depth': 4, 'n_estimators': 231, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.6778661833429002, 'min_weight_fraction_leaf': 0.35612645124522524}. Best is trial 15 with value: 0.7572849484519277.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.8170212765957446
Fold 4 C-index: 0.7224334600760456
Fold 5 C-index: 0.6995708154506438
[I 2024-04-16 19:10:46,874] Trial 17 finished with value: 0.7059353800436845 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 11, 'min_samples_leaf': 14, 'max_depth': 9, 'n_estimators': 100, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.3861349103797888, 'min_weight_fraction_leaf': 0.0896057534547

Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.8723404255319149
Fold 4 C-index: 0.7414448669201521
Fold 5 C-index: 0.7424892703862661
[I 2024-04-16 19:10:56,464] Trial 31 finished with value: 0.7450836287108459 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 8, 'max_depth': 1, 'n_estimators': 89, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.6115041201105692, 'min_weight_fraction_leaf': 0.07529457384788989}. Best is trial 24 with value: 0.7999244832738667.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.8217054263565892
Fold 3 C-index: 0.8765957446808511
Fold 4 C-index: 0.8098859315589354
Fold 5 C-index: 0.8240343347639485
[I 2024-04-16 19:10:57,149] Trial 32 finished with value: 0.7891534508186784 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 6, 'max_depth': 2, 'n_estimators': 106, 'oob_score': False, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.24189951983836425
Fold 2 IBS: 0.22116291305449884
Fold 3 IBS: 0.21713072638394984
Fold 4 IBS: 0.24023416048014104
Fold 5 IBS: 0.22691624187044754
[I 2024-04-16 19:13:47,993] Trial 30 finished with value: 0.2294687123254803 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 8, 'min_samples_leaf': 7, 'max_depth': 5, 'n_estimators': 180, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.5988223393644435, 'min_weight_fraction_leaf': 0.20241101766143602}. Best is trial 15 with value: 0.21555098369824366.
Fold 1 IBS: 0.24364620279536037
Fold 2 IBS: 0.2205505651382432
Fold 3 IBS: 0.21184589055027794
Fold 4 IBS: 0.24142198209294274
Fold 5 IBS: 0.2268782233658812
[I 2024-04-16 19:13:49,755] Trial 31 finished with value: 0.22886857278854106 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 13, 'min_samples_leaf': 11, 'max_depth': 8, 'n_estimators': 162, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0

Fold 1 IBS: 0.24970180838617465
Fold 2 IBS: 0.20711640796461644
Fold 3 IBS: 0.210622445605441
Fold 4 IBS: 0.2398038048480638
Fold 5 IBS: 0.23687430367329662
[I 2024-04-16 19:15:03,383] Trial 45 finished with value: 0.22882375409551847 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 19, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 481, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.6241818506764952, 'min_weight_fraction_leaf': 0.003649071529300437}. Best is trial 41 with value: 0.2125643090401542.
Fold 1 IBS: 0.23201805390304453
Fold 2 IBS: 0.20002733588165444
Fold 3 IBS: 0.19908476397700547
Fold 4 IBS: 0.22074033459452136
Fold 5 IBS: 0.21494542822340232
[I 2024-04-16 19:15:12,982] Trial 46 finished with value: 0.2133631833159256 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 2, 'max_depth': 15, 'n_estimators': 419, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.55

Fold 1 IBS: 0.24516656524539981
Fold 2 IBS: 0.21062905264741455
Fold 3 IBS: 0.20928713648563754
Fold 4 IBS: 0.24177268946133224
Fold 5 IBS: 0.23349939521162735
[I 2024-04-16 19:16:26,966] Trial 60 finished with value: 0.2280709678102823 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 13, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 382, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.7740796078428375, 'min_weight_fraction_leaf': 0.004254061138558238}. Best is trial 59 with value: 0.21106902851963083.
Fold 1 IBS: 0.23239287585025117
Fold 2 IBS: 0.19481155216054105
Fold 3 IBS: 0.19221809734433284
Fold 4 IBS: 0.21072513374212
Fold 5 IBS: 0.21430420045958357
[I 2024-04-16 19:16:31,305] Trial 61 finished with value: 0.20889037191136572 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 14, 'min_samples_leaf': 9, 'max_depth': 12, 'n_estimators': 258, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.

Fold 1 IBS: 0.23374241288259937
Fold 2 IBS: 0.19930439672774886
Fold 3 IBS: 0.18717738406829976
Fold 4 IBS: 0.20823176235963525
Fold 5 IBS: 0.21501371316102338
[I 2024-04-16 19:18:09,858] Trial 75 finished with value: 0.20869393383986132 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 11, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 435, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9528867121166885, 'min_weight_fraction_leaf': 0.0012590377645196535}. Best is trial 73 with value: 0.20839397477875155.
Fold 1 IBS: 0.24525891729558977
Fold 2 IBS: 0.21896003468791347
Fold 3 IBS: 0.21598508493564658
Fold 4 IBS: 0.24394612844374725
Fold 5 IBS: 0.2322350643210711
[I 2024-04-16 19:18:14,514] Trial 76 finished with value: 0.23127704593679366 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 11, 'min_samples_leaf': 13, 'max_depth': 11, 'n_estimators': 465, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples

Fold 1 IBS: 0.24308229605873613
Fold 2 IBS: 0.20243085243083073
Fold 3 IBS: 0.20257300733372727
Fold 4 IBS: 0.2350012477048381
Fold 5 IBS: 0.22812847237666628
[I 2024-04-16 19:19:50,815] Trial 90 finished with value: 0.2222431751809597 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 7, 'min_samples_leaf': 9, 'max_depth': 14, 'n_estimators': 427, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.9384169100812306, 'min_weight_fraction_leaf': 0.10055258156251395}. Best is trial 82 with value: 0.20797277264373898.
Fold 1 IBS: 0.23494236000327381
Fold 2 IBS: 0.19851393414446025
Fold 3 IBS: 0.188316054669442
Fold 4 IBS: 0.20779679335240808
Fold 5 IBS: 0.21280969619087245
[I 2024-04-16 19:20:00,821] Trial 91 finished with value: 0.20847576767209133 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 8, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 442, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.961995

In [64]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [65]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.842
train_ibs:  0.208


#### Test

In [66]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [67]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=16, max_features=None, max_leaf_nodes=10,
                   max_samples=0.9186714469202719, min_samples_leaf=2,
                   min_samples_split=3,
                   min_weight_fraction_leaf=0.04956115599708024,
                   n_estimators=474, random_state=123, warm_start=True)

C-index score: 0.567


ExtraSurvivalTrees(max_depth=12, max_features=None, max_leaf_nodes=9,
                   max_samples=0.9407754324156289, min_samples_leaf=9,
                   min_samples_split=2,
                   min_weight_fraction_leaf=0.04206241234084829,
                   n_estimators=431, oob_score=True, random_state=123)

IBS: 0.235


In [68]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [69]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 19:21:05,825] A new study created in memory with name: no-name-9e099d4f-0adb-410b-91ab-71812c417c3a


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 19:21:31,389] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 19:21:44,192] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 19:28:11,833] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.6841431930275611.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 19:28:48,096] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 19:36:19,545] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 12 with value: 0.6841431930275611.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 19:36:39,245] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedm

...

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.22939559304809248
[I 2024-04-16 21:02:05,226] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.809323041898869, 'learning_rate': 0.0040826970179266685, 'dropout_rate': 0.36065535241543395, 'n_estimators': 435, 'criterion': 'squared_error', 'ccp_alpha': 1.5126136571072866, 'min_weight_fraction_leaf': 0.2002957617254777, 'max_features': 'auto', 'min_impurity_decrease': 2.0868015436948727e-05, 'validation_fraction': 0.8814649292460881, 'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2}. Best is trial 41 with value: 0.23455338278002208.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-16 21:03:06,151] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.2818850250060197, 'learning_rate': 0.0985092

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 21:19:09,107] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8489175059797514, 'learning_rate': 0.008448226967076113, 'dropout_rate': 0.12725750793823018, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 0.893877768639663, 'min_weight_fraction_leaf': 0.2702972473075093, 'max_features': 'auto', 'min_impurity_decrease': 7.586810829424823e-05, 'validation_fraction': 0.8971088342813041, 'min_samples_split': 17, 'max_leaf_nodes': 17, 'min_samples_leaf': 8, 'max_depth': 4}. Best is trial 62 with value: 0.23436365650963703.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792296
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 21:22:19,003] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9594402941587806,

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-16 21:39:54,669] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7234820637720293, 'learning_rate': 0.04128077713454106, 'dropout_rate': 0.19884799526452568, 'n_estimators': 302, 'criterion': 'squared_error', 'ccp_alpha': 1.2832467794713243, 'min_weight_fraction_leaf': 0.4027406894595495, 'max_features': None, 'min_impurity_decrease': 5.35591590419612e-07, 'validation_fraction': 0.8758106078120852, 'min_samples_split': 19, 'max_leaf_nodes': 5, 'min_samples_leaf': 10, 'max_depth': 2}. Best is trial 70 with value: 0.23410964446224067.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 21:41:26,795] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6708593402641205, '

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-16 21:50:51,630] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6879348852579991, 'learning_rate': 0.08209035744510738, 'dropout_rate': 0.24522201083374043, 'n_estimators': 84, 'criterion': 'squared_error', 'ccp_alpha': 0.2983993026610602, 'min_weight_fraction_leaf': 0.3774672254272644, 'max_features': None, 'min_impurity_decrease': 5.4011226362110275e-06, 'validation_fraction': 0.568843188657281, 'min_samples_split': 16, 'max_leaf_nodes': 6, 'min_samples_leaf': 4, 'max_depth': 11}. Best is trial 81 with value: 0.23062671367871226.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 21:51:01,254] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6555693108457097, 'learning_rate': 0.08348043724232

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 21:57:18,888] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.4750010457704055, 'learning_rate': 0.07813501740629669, 'dropout_rate': 0.30318261654818046, 'n_estimators': 242, 'criterion': 'squared_error', 'ccp_alpha': 1.7079733343503498, 'min_weight_fraction_leaf': 0.32269779176244495, 'max_features': 0.1, 'min_impurity_decrease': 1.276230329676773e-06, 'validation_fraction': 0.36954438153697, 'min_samples_split': 11, 'max_leaf_nodes': 9, 'min_samples_leaf': 7, 'max_depth': 2}. Best is trial 91 with value: 0.2300644165494142.


* Best trial for IBS: 
 FrozenTrial(number=91, state=TrialState.COMPLETE, values=[0.2300644165494142], datetime_start=datetime.datetime(2024, 4, 16, 21, 51, 16, 171522), datetime_complete=datetime.datetime(2024, 4, 16, 21, 52, 12, 539211), params={'subsample': 0.579213280645811, 'learning_rate': 0.09426091382038485, 'dropout_rate': 0.18479264516541616,

In [70]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [71]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.7
train_ibs:  0.23


#### Test

In [72]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [73]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.004507507039685693,
                                 criterion='squared_error',
                                 dropout_rate=0.8022681311855436,
                                 learning_rate=0.010888929954369636,
                                 max_depth=12, max_features='auto',
                                 max_leaf_nodes=3,
                                 min_impurity_decrease=4.0700060224631717e-07,
                                 min_samples_leaf=10, min_samples_split=19,
                                 min_weight_fraction_leaf=0.1685851039915477,
                                 n_estimators=443, random_state=123,
                                 subsample=0.8597838031871279,
                                 validation_fraction=0.8827408742810589)

C-index score: 0.516


GradientBoostingSurvivalAnalysis(ccp_alpha=0.03787089343697578,
                                 criterion='squared_error',
                                 dropout_rate=0.18479264516541616,
                                 learning_rate=0.09426091382038485, max_depth=2,
                                 max_leaf_nodes=6,
                                 min_impurity_decrease=7.4293259533530295e-06,
                                 min_samples_leaf=7, min_samples_split=18,
                                 min_weight_fraction_leaf=0.4036747584607865,
                                 n_estimators=390, random_state=123,
                                 subsample=0.579213280645811,
                                 validation_fraction=0.5325834925366492)

IBS: 0.228


In [74]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [75]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [76]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 21:57:41,206] A new study created in memory with name: no-name-4622412a-4835-4172-8a85-da25fc5e1352


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6523605150214592
[I 2024-04-16 21:57:49,854] Trial 0 finished with value: 0.6232514766080982 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6232514766080982.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6523605150214592
[I 2024-04-16 21:58:27,828] Trial 1 finished with value: 0.6232514766080982 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6232514766080982.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6577946768060836
Fold 5 C-index: 0.6738197424892703
[I 2024-04-16 22:04:59,403] Trial 19 finished with value: 0.6352321782336097 and parameters: {'subsample': 0.2331417557263912, 'dropout_rate': 0.6703689925392199, 'n_estimators': 431, 'learning_rate': 0.07786289506257674}. Best is trial 18 with value: 0.6437467548997808.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6615969581749049
Fold 5 C-index: 0.6824034334763949
[I 2024-04-16 22:05:27,024] Trial 20 finished with value: 0.6385278044043572 and parameters: {'subsample': 0.23177629557063834, 'dropout_rate': 0.4044384358521501, 'n_estimators': 423, 'learning_rate': 0.0817175642960023}. Best is trial 18 with value: 0.6437467548997808.
Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.5574468085106383
F

Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6501901140684411
Fold 5 C-index: 0.6781115879828327
[I 2024-04-16 22:12:57,986] Trial 38 finished with value: 0.6337728220357896 and parameters: {'subsample': 0.31616457114097773, 'dropout_rate': 0.3046686418127724, 'n_estimators': 448, 'learning_rate': 0.06884463570078815}. Best is trial 32 with value: 0.6439425805496771.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6781115879828327
[I 2024-04-16 22:13:13,852] Trial 39 finished with value: 0.6399871606021772 and parameters: {'subsample': 0.22343114379949888, 'dropout_rate': 0.46838390741669006, 'n_estimators': 278, 'learning_rate': 0.05598015324329953}. Best is trial 32 with value: 0.6439425805496771.
Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.561702127659574

...

Fold 3 IBS: 0.27495840642227154
Fold 4 IBS: 0.2403770465027076
Fold 5 IBS: 0.22344734604030783
[I 2024-04-16 23:12:02,834] Trial 19 finished with value: 0.24199914104647421 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.24532728078212304
Fold 2 IBS: 0.21338053568843193
Fold 3 IBS: 0.23280583204014674
Fold 4 IBS: 0.23050963160208696
Fold 5 IBS: 0.21229213294998353
[I 2024-04-16 23:12:06,722] Trial 20 finished with value: 0.22686308261255445 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.24480948267779246
Fold 2 IBS: 0.217894217650721
Fold 3 IBS: 0.23043919128917217
Fold 4 IBS: 0.23275995527788665
Fold 5 IBS: 0.215372968667489
[I 2024-04-16 23:12:10,522] Trial 21 finis

Fold 4 IBS: 0.23376225132374828
Fold 5 IBS: 0.21475575962300342
[I 2024-04-16 23:13:54,808] Trial 38 finished with value: 0.23328897274995045 and parameters: {'subsample': 0.605229739689187, 'dropout_rate': 0.13272164755980653, 'n_estimators': 176, 'learning_rate': 0.01283698591663533}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.29108106719697524
Fold 2 IBS: 0.21415810554809628
Fold 3 IBS: 0.2917820507930866
Fold 4 IBS: 0.2588982634047282
Fold 5 IBS: 0.24407574900323586
[I 2024-04-16 23:13:57,899] Trial 39 finished with value: 0.25999904718922445 and parameters: {'subsample': 0.6970162917669863, 'dropout_rate': 0.48841341315505027, 'n_estimators': 59, 'learning_rate': 0.06892741183938003}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.2925659515921918
Fold 2 IBS: 0.2140746731129848
Fold 3 IBS: 0.2919432513336724
Fold 4 IBS: 0.257477121073266
Fold 5 IBS: 0.24657769529804316
[I 2024-04-16 23:14:03,695] Trial 40 finished with value: 0.2605277384820316 

Fold 5 IBS: 0.27450442032616973
[I 2024-04-16 23:15:12,824] Trial 57 finished with value: 0.29085967593170914 and parameters: {'subsample': 0.763955514973389, 'dropout_rate': 0.9964559295946985, 'n_estimators': 282, 'learning_rate': 0.02203371126489022}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.2514876544143989
Fold 2 IBS: 0.20473165445913052
Fold 3 IBS: 0.2442786298594191
Fold 4 IBS: 0.2292932477615486
Fold 5 IBS: 0.2122783863810015
[I 2024-04-16 23:15:15,474] Trial 58 finished with value: 0.2284139145750997 and parameters: {'subsample': 0.8210665501105633, 'dropout_rate': 0.9557982366790785, 'n_estimators': 50, 'learning_rate': 0.03153701184110286}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.2450009034050451
Fold 2 IBS: 0.21574021216390366
Fold 3 IBS: 0.2316126494096215
Fold 4 IBS: 0.2313085066143514
Fold 5 IBS: 0.2135215585555951
[I 2024-04-16 23:15:16,838] Trial 59 finished with value: 0.22743676602970336 and parameters: {'subsample': 0.7

Fold 5 IBS: 0.21457551785343906
[I 2024-04-16 23:17:00,366] Trial 76 finished with value: 0.2278190214501346 and parameters: {'subsample': 0.5683268745859826, 'dropout_rate': 0.8653081987830238, 'n_estimators': 34, 'learning_rate': 0.020347121996093395}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.24975605957417094
Fold 2 IBS: 0.20544669926019324
Fold 3 IBS: 0.24292589881032364
Fold 4 IBS: 0.2283240329592914
Fold 5 IBS: 0.21054396005104126
[I 2024-04-16 23:17:07,594] Trial 77 finished with value: 0.22739933013100408 and parameters: {'subsample': 0.5096986308003991, 'dropout_rate': 0.49881327862855496, 'n_estimators': 145, 'learning_rate': 0.010267257888313383}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.24684927097070503
Fold 2 IBS: 0.2094952280230178
Fold 3 IBS: 0.23652719105524916
Fold 4 IBS: 0.22913951054643386
Fold 5 IBS: 0.2106573202990221
[I 2024-04-16 23:17:11,607] Trial 78 finished with value: 0.2265337041788856 and parameters: {'subsamp

Fold 5 IBS: 0.21018857248815617
[I 2024-04-16 23:18:34,121] Trial 95 finished with value: 0.22600775713838323 and parameters: {'subsample': 0.37798247189331685, 'dropout_rate': 0.8572208063796695, 'n_estimators': 112, 'learning_rate': 0.01063058308878333}. Best is trial 85 with value: 0.22596006011310088.
Fold 1 IBS: 0.2471425193657016
Fold 2 IBS: 0.20576111765918625
Fold 3 IBS: 0.2415748835282002
Fold 4 IBS: 0.22741639083301954
Fold 5 IBS: 0.2095395501057605
[I 2024-04-16 23:18:39,560] Trial 96 finished with value: 0.22628689229837362 and parameters: {'subsample': 0.34523349906195283, 'dropout_rate': 0.7739280807288145, 'n_estimators': 132, 'learning_rate': 0.010648288131411126}. Best is trial 85 with value: 0.22596006011310088.
Fold 1 IBS: 0.24385115968166024
Fold 2 IBS: 0.21643646571599923
Fold 3 IBS: 0.23116557489710682
Fold 4 IBS: 0.23177773853470693
Fold 5 IBS: 0.21519967612969007
[I 2024-04-16 23:18:46,640] Trial 97 finished with value: 0.22768612299183263 and parameters: {'subs

In [77]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [78]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.646
train_ibs:  0.224


#### Test

In [79]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [80]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10106238648390999,
                                              learning_rate=0.08108895994342519,
                                              n_estimators=425,
                                              random_state=123,
                                              subsample=0.12866773796450384)

C-index score: 0.534


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.7782160113329367,
                                              learning_rate=0.005308151321564225,
                                              n_estimators=212,
                                              random_state=123,
                                              subsample=0.17778434918103095)

IBS: 0.233


In [81]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [82]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.847,1.0
ExtraSurvivalTrees,0.842,2.0
GradientBoosting,0.700,3.0
ComponentwiseGradientBoosting,0.646,4.0
CoxLasso,0.574,5.5
CoxElastic,0.574,5.5
CoxRidge,0.559,7.0


In [83]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.208,1.0
Randomsurvivalforest,0.212,2.0
ComponentwiseGradientBoosting,0.224,3.0
GradientBoosting,0.230,4.0
CoxRidge,0.236,6.0
CoxLasso,0.236,6.0
CoxElastic,0.236,6.0


In [84]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.567,1.0
CoxLasso,0.562,2.5
CoxElastic,0.562,2.5
CoxRidge,0.546,4.0
Randomsurvivalforest,0.540,5.0
ComponentwiseGradientBoosting,0.534,6.0
GradientBoosting,0.516,7.0


In [85]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
GradientBoosting,0.228,1.0
CoxRidge,0.229,3.0
CoxLasso,0.229,3.0
CoxElastic,0.229,3.0
ComponentwiseGradientBoosting,0.233,5.0
ExtraSurvivalTrees,0.235,6.0
Randomsurvivalforest,0.262,7.0


In [90]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/dfs/robust/no_selection/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_dfs_robust_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [91]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-17
